In [3]:
import os
import cv2
import numpy as np

def slice_image_and_labels(
    img_path,
    label_path,
    out_img_dir,
    out_lbl_dir,
    tile_size=640,
    overlap_ratio=0.2,
):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    img = cv2.imread(img_path)
    if img is None:
        return
    h, w, _ = img.shape
    base_name = os.path.splitext(os.path.basename(img_path))[0]

    boxes = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue
                cls, cx, cy, bw, bh = map(float, parts)
                # Absolute pixel coordinates
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                x2 = (cx + bw / 2) * w
                y2 = (cy + bh / 2) * h
                boxes.append([int(cls), x1, y1, x2, y2])

    step = int(tile_size * (1 - overlap_ratio))

    for y in range(0, h, step):
        for x in range(0, w, step):
            x_end = min(x + tile_size, w)
            y_end = min(y + tile_size, h)
            x_start = max(0, x_end - tile_size)
            y_start = max(0, y_end - tile_size)

            tile = img[y_start:y_end, x_start:x_end]
            tile_boxes = []

            for cls, bx1, by1, bx2, by2 in boxes:
                bcx = (bx1 + bx2) / 2
                bcy = (by1 + by2) / 2

                # Quantize target centers to active frame
                if x_start <= bcx < x_end and y_start <= bcy < y_end:
                    # Clip coordinates safely to current window constraints
                    tx1 = max(0, bx1 - x_start)
                    ty1 = max(0, by1 - y_start)
                    tx2 = min(tile_size, bx2 - x_start)
                    ty2 = min(tile_size, by2 - y_start)

                    # Normalized coordinates relative to the 640x640 tile size
                    tcx = ((tx1 + tx2) / 2) / tile_size
                    tcy = ((ty1 + ty2) / 2) / tile_size
                    tbw = (tx2 - tx1) / tile_size
                    tbh = (ty2 - ty1) / tile_size

                    tile_boxes.append(f'{cls} {tcx:.6f} {tcy:.6f} {tbw:.6f} {tbh:.6f}')

            tile_id = f'{base_name}_x{x_start}_y{y_start}'
            
            # Save if objects are present or randomly include background sample (~10%)
            if len(tile_boxes) > 0 or np.random.rand() < 0.10:
                cv2.imwrite(os.path.join(out_img_dir, f'{tile_id}.jpg'), tile)
                with open(os.path.join(out_lbl_dir, f'{tile_id}.txt'), 'w') as f_out:
                    f_out.write('\n'.join(tile_boxes))


In [4]:
import glob

def slice_entire_dataset(src_img_dir, src_lbl_dir, out_img_dir, out_lbl_dir, **kwargs):
    """
    Loops through all images in src_img_dir, finds their matching label 
    files in src_lbl_dir, and processes them one-by-one.
    """
    # Find all common image formats (jpg, jpeg, png)
    img_extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    img_paths = []
    for ext in img_extensions:
        img_paths.extend(glob.glob(os.path.join(src_img_dir, ext)))
        
    print(f"Found {len(img_paths)} images to process.")

    for img_path in img_paths:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        
        # Match corresponding .txt YOLO label file
        label_path = os.path.join(src_lbl_dir, f"{base_name}.txt")
        
        # Run the one-image-at-a-time slicer
        slice_image_and_labels(
            img_path=img_path,
            label_path=label_path,
            out_img_dir=out_img_dir,
            out_lbl_dir=out_lbl_dir,
            **kwargs
        )
        
    print("Slicing complete!")

# Example usage:
# slice_entire_dataset("data/train/images", "data/train/labels", "sliced/images", "sliced/labels", tile_size=640)


In [ ]:
slice_entire_dataset(src_img_dir="", src_lbl_dir="", out_img_dir="", out_lbl_dir="")